# Strato — a solar-electric stratospheric airship (HAPS)

Can a helium airship powered by the sun hold station at **18–22 km over Europe** for weeks or months with a
communications or observation payload — and what limits it? This notebook walks one design through Vegeta's Stage V8
models; the full study (optimisation of 15–120 m ships, Pareto fronts, sensitivity, seasons) is the campaign
`labs/aero/campaigns/strato/run.py`, whose report is `campaigns/strato/results/report.md`.

```
og.stratosphere, og.lifting_gas, og.solar, og.envelope   (Ogma: verified physics, tested against reference values)
strato_geometry ─► strato (Dedalus CAD) ─► STEP / pictures / 3D page
strato_design (gas fill, superpressure, plies, mass budget, trim) ─► strato_sim (day/night, endurance, ascent)
                                                                  └► strato_feasibility (the gate) ─► strato_study
```
Assumptions and the verified/estimate split: `designs/STRATO.md`. Units SI; CAD in mm (Dedalus).

In [ ]:
DEBUG = True
from vegeta.labkit import bootstrap
DEBUG = bootstrap.lab("aero", DEBUG)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ogma import stratosphere as st
import strato_geometry as sg, strato, strato_design as sd, strato_feasibility as sf, strato_sim as ss, strato_wind as sw

## The air at 20 km (USSA 1976)

In [ ]:
for z in (0, 11000, 18000, 20000, 22000):
    s = st.state(z)
    print(f"{z/1000:5.1f} km  T {s.temperature:6.2f} K  p {s.pressure:9.1f} Pa  rho {s.density:.5f} kg/m3  g {s.gravity:.4f}")

## Geometry and CAD: the 15 m × 3.5 m demonstrator, scaled to 80 m

In [ ]:
for L, m in sg.family().items():
    print({k: round(v, 2) for k, v in m.summary().items()})

In [ ]:
from vegeta.dedalus import viz
parts = strato.parts(**strato.for_length(15.0))
pl = viz.plot3d(*parts.values(), colors=["#d9dde3", "#e07a5f", "#3d405b", "#f2cc8f", "#1d3557", "#81b29a", "#9a8c98"])
viz.show(pl)

## One design through the gate

The 80 m reference of the goldens (`tests/test_strato_golden.py`): fineness 3.5, Zylon laminate, GaAs array, a
sealed superpressure hull with a 150 Pa night reserve.

In [ ]:
d = sd.Design(length=80.0, fineness=3.5, payload_mass=50.0, battery_energy=40000.0, propulsion_power=30000.0,
              laminate="zylon", fin_scale=0.7, night_superpressure=150.0)
for month in (7, 12):
    r = sf.evaluate(d, sd.Mission(month=month))
    print(month, "feasible" if r["feasible"] else r["problems"], {k: round(r[k], 2) for k in
          ("system_mass", "net_lift", "surplus_lift", "superheat_day", "dp_max", "availability", "gas_life_days")})
    print("   ", r["warnings"])

In [ ]:
r = sf.evaluate(d, sd.Mission(month=7), detail=True)
cyc = r["cycle"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(cyc["t"] / 3600, cyc["solar"] / 1e3, label="array [kW]"); ax[0].plot(cyc["t"] / 3600, (cyc["load"] + cyc["prop"]) / 1e3, label="loads + propulsion [kW]")
ax[1].plot(cyc["t"] / 3600, cyc["superheat"], label="superheat [K]"); ax[1].plot(cyc["t"] / 3600, cyc["dp"] / 10, label="superpressure [daPa]")
for a in ax: a.set_xlabel("hours, 15 July (UTC)"); a.legend()
plt.show()

## Ascent and a week on station

In [ ]:
a = ss.ascent(r["configuration"], dt=4.0)
print({k: round(a[k], 2) for k in ("float_altitude", "max_rate", "time_to_99pct_s", "drift_km", "dp_top")})
e = ss.endurance(r["configuration"], days=7, dt=1800.0)
print({k: e[k] for k in ("on_station", "lost_at_s", "gas_lost_kg", "max_dist_km", "min_altitude")})

## The study

Run `python labs/aero/campaigns/strato/run.py` (about two hours on four cores; `--quick` for a smoke run) and read
`campaigns/strato/results/report.md`; the interactive 3D page is `campaigns/strato/cad/strato_3d.html`
(`viewer.py`).

In [ ]:
from pathlib import Path
report = Path(strato.__file__).resolve().parents[1] / "campaigns" / "strato" / "results" / "report.md"
print(report.read_text()[:3000] if report.exists() else f"run the campaign first: {report}")